In [0]:
train_table = "workspace.default.customer_churn_dataset_training_master"
test_table = "workspace.default.customer_churn_dataset_testing_master"

train_df = spark.table(train_table) 
test_df = spark.table(test_table)

print("Train row count:", train_df.count())
print("Test row count:", test_df.count())

display(train_df.limit(5))
display(test_df.limit(5))

Train row count: 440833
Test row count: 505207


CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
2,30,Female,39,14,5,18,Standard,Annual,932.0,17,1
3,65,Female,49,1,10,8,Basic,Monthly,557.0,6,1
4,55,Female,14,4,6,18,Basic,Quarterly,185.0,3,1
5,58,Male,38,21,7,7,Standard,Monthly,396.0,29,1
6,23,Male,32,20,5,8,Basic,Monthly,617.0,20,1


CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
1,22,Female,25,14,4,27,Basic,Monthly,598.0,9,1
2,41,Female,28,28,7,13,Standard,Monthly,584.0,20,0
3,47,Male,27,10,2,29,Premium,Annual,757.0,21,0
4,35,Male,9,12,5,17,Premium,Quarterly,232.0,18,0
5,53,Female,58,24,9,2,Standard,Annual,533.0,18,0


In [0]:
print("TRAIN SCHEMA")
train_df.printSchema()

print("TEST SCHEMA")
test_df.printSchema()

print("TRAIN COLUMNS")
print(train_df.columns)

print("TEST COLUMNS")
print(test_df.columns)

TRAIN SCHEMA
root
 |-- CustomerID: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Tenure: long (nullable = true)
 |-- Usage Frequency: long (nullable = true)
 |-- Support Calls: long (nullable = true)
 |-- Payment Delay: long (nullable = true)
 |-- Subscription Type: string (nullable = true)
 |-- Contract Length: string (nullable = true)
 |-- Total Spend: double (nullable = true)
 |-- Last Interaction: long (nullable = true)
 |-- Churn: long (nullable = true)

TEST SCHEMA
root
 |-- CustomerID: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Tenure: long (nullable = true)
 |-- Usage Frequency: long (nullable = true)
 |-- Support Calls: long (nullable = true)
 |-- Payment Delay: long (nullable = true)
 |-- Subscription Type: string (nullable = true)
 |-- Contract Length: string (nullable = true)
 |-- Total Spend: double (nullable = true)
 |-- Last Interaction: long (nullable = true)
 |-

In [0]:
from pyspark.sql import functions as F

def clean_spark_columns(df):
    for old_name in df.columns:
        new_name = old_name.strip().lower().replace(" ", "_")
        df = df.withColumnRenamed(old_name, new_name)
    return df

train_df = clean_spark_columns(train_df)
test_df = clean_spark_columns(test_df)

print(train_df.columns)
print(test_df.columns)

['customerid', 'age', 'gender', 'tenure', 'usage_frequency', 'support_calls', 'payment_delay', 'subscription_type', 'contract_length', 'total_spend', 'last_interaction', 'churn']
['customerid', 'age', 'gender', 'tenure', 'usage_frequency', 'support_calls', 'payment_delay', 'subscription_type', 'contract_length', 'total_spend', 'last_interaction', 'churn']


In [0]:
display(train_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in train_df.columns
]))

display(test_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in test_df.columns
]))

customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
1,1,1,1,1,1,1,1,1,1,1,1


customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
1,1,1,1,1,1,1,1,1,1,1,1


In [0]:
numeric_cols = [
    "customerid",
    "age",
    "tenure",
    "usage_frequency",
    "support_calls",
    "payment_delay",
    "total_spend",
    "last_interaction",
    "churn"
]

categorical_cols = [
    "gender",
    "subscription_type",
    "contract_length"
]

for c in numeric_cols:
    train_df = train_df.withColumn(c, F.col(c).cast("double"))
    test_df = test_df.withColumn(c, F.col(c).cast("double"))

for c in categorical_cols:
    train_df = train_df.withColumn(c, F.trim(F.col(c).cast("string")))
    test_df = test_df.withColumn(c, F.trim(F.col(c).cast("string")))

train_df = train_df.dropna(subset=["churn"])
test_df = test_df.dropna(subset=["churn"])

for c in numeric_cols:
    train_df = train_df.fillna({c: 0})
    test_df = test_df.fillna({c: 0})

for c in categorical_cols:
    train_df = train_df.fillna({c: "Unknown"})
    test_df = test_df.fillna({c: "Unknown"})

display(train_df.limit(5))
display(test_df.limit(5))

customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4.0,35.0,Male,9.0,12.0,5.0,17.0,Premium,Quarterly,232.0,18.0,0.0
5.0,53.0,Female,58.0,24.0,9.0,2.0,Standard,Annual,533.0,18.0,0.0


In [0]:
int_cols = [
    "customerid",
    "age",
    "tenure",
    "usage_frequency",
    "support_calls",
    "payment_delay",
    "total_spend",
    "last_interaction",
    "churn"
]

for c in int_cols:
    train_df = train_df.withColumn(c, F.round(F.col(c)).cast("int"))
    test_df = test_df.withColumn(c, F.round(F.col(c)).cast("int"))

train_df.printSchema()
test_df.printSchema()

root
 |-- customerid: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- tenure: integer (nullable = true)
 |-- usage_frequency: integer (nullable = true)
 |-- support_calls: integer (nullable = true)
 |-- payment_delay: integer (nullable = true)
 |-- subscription_type: string (nullable = false)
 |-- contract_length: string (nullable = false)
 |-- total_spend: integer (nullable = true)
 |-- last_interaction: integer (nullable = true)
 |-- churn: integer (nullable = true)

root
 |-- customerid: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- tenure: integer (nullable = true)
 |-- usage_frequency: integer (nullable = true)
 |-- support_calls: integer (nullable = true)
 |-- payment_delay: integer (nullable = true)
 |-- subscription_type: string (nullable = false)
 |-- contract_length: string (nullable = false)
 |-- total_spend: integer (nullable = true)
 |-- last_interaction: i

In [0]:
display(
    train_df.groupBy("churn").count().orderBy("churn")
)

display(
    test_df.groupBy("churn").count().orderBy("churn")
)

churn,count
0,190833
1,249999


churn,count
0,224714
1,280492


In [0]:
clean_train_table = "workspace.default.customer_churn_train_clean"
clean_test_table = "workspace.default.customer_churn_test_clean"

train_df.write.mode("overwrite").saveAsTable(clean_train_table)
test_df.write.mode("overwrite").saveAsTable(clean_test_table)

print("Saved:", clean_train_table)
print("Saved:", clean_test_table)

Saved: workspace.default.customer_churn_train_clean
Saved: workspace.default.customer_churn_test_clean


In [0]:
train_df_with_source = train_df.withColumn("dataset_type", F.lit("train"))
test_df_with_source = test_df.withColumn("dataset_type", F.lit("test"))

full_df = train_df_with_source.unionByName(test_df_with_source)

full_table = "workspace.default.customer_churn_full"
full_df.write.mode("overwrite").saveAsTable(full_table)

display(full_df.groupBy("dataset_type").count())

dataset_type,count
train,440832
test,505206


In [0]:
%sql
SELECT
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full;

churn_rate_pct
56.08


In [0]:
%sql
SELECT
  gender,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY gender
ORDER BY churn_rate_pct DESC;

gender,customers,churn_rate_pct
Female,415513,65.71
Male,530525,48.53


In [0]:
%sql
SELECT
  subscription_type,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY subscription_type
ORDER BY churn_rate_pct DESC;

subscription_type,customers,churn_rate_pct
Basic,307503,57.49
Standard,319758,55.48
Premium,318777,55.31


In [0]:
%sql
SELECT
  subscription_type,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY subscription_type
ORDER BY churn_rate_pct DESC;

subscription_type,customers,churn_rate_pct
Basic,307503,57.49
Standard,319758,55.48
Premium,318777,55.31


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
  CASE
    WHEN support_calls = 0 THEN '0'
    WHEN support_calls BETWEEN 1 AND 2 THEN '1-2'
    WHEN support_calls BETWEEN 3 AND 5 THEN '3-5'
    ELSE '6+'
  END AS support_call_bucket,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY support_call_bucket
ORDER BY support_call_bucket;

support_call_bucket,customers,churn_rate_pct
0,144717,30.07
1-2,281835,30.67
3-5,249322,57.47
6+,270164,95.22


In [0]:
%sql
SELECT
  CASE
    WHEN payment_delay = 0 THEN '0'
    WHEN payment_delay BETWEEN 1 AND 5 THEN '1-5'
    WHEN payment_delay BETWEEN 6 AND 10 THEN '6-10'
    ELSE '11+'
  END AS payment_delay_bucket,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY payment_delay_bucket
ORDER BY payment_delay_bucket;

payment_delay_bucket,customers,churn_rate_pct
0,35402,45.0
1-5,176869,44.82
11+,556362,63.9
6-10,177405,44.97


In [0]:
%sql
SELECT
  CASE
    WHEN tenure < 6 THEN '<6'
    WHEN tenure < 12 THEN '6-11'
    WHEN tenure < 24 THEN '12-23'
    ELSE '24+'
  END AS tenure_bucket,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY tenure_bucket
ORDER BY tenure_bucket;

tenure_bucket,customers,churn_rate_pct
12-23,171068,60.93
24+,607535,54.5
6-11,97316,52.79
<6,70119,62.42


In [0]:
%sql
SELECT
  CASE
    WHEN usage_frequency <= 5 THEN '0-5'
    WHEN usage_frequency <= 10 THEN '6-10'
    WHEN usage_frequency <= 15 THEN '11-15'
    ELSE '16+'
  END AS usage_bucket,
  COUNT(*) AS customers,
  ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM workspace.default.customer_churn_full
GROUP BY usage_bucket
ORDER BY usage_bucket;

usage_bucket,customers,churn_rate_pct
0-5,149576,61.75
11-15,161876,54.15
16+,484502,54.16
6-10,150084,58.67


In [0]:
train_pdf = train_df.toPandas()
test_pdf = test_df.toPandas()

print(train_pdf.shape)
print(test_pdf.shape)
print(train_pdf.head())

(440832, 12)
(505206, 12)
   customerid  age  gender  ...  total_spend  last_interaction  churn
0           2   30  Female  ...          932                17      1
1           3   65  Female  ...          557                 6      1
2           4   55  Female  ...          185                 3      1
3           5   58    Male  ...          396                29      1
4           6   23    Male  ...          617                20      1

[5 rows x 12 columns]


In [0]:
feature_cols = [
    "age",
    "gender",
    "tenure",
    "usage_frequency",
    "support_calls",
    "payment_delay",
    "subscription_type",
    "contract_length",
    "total_spend",
    "last_interaction"
]

target_col = "churn"

train_ml = train_pdf.copy()
test_ml = test_pdf.copy()

In [0]:
import pandas as pd

full_for_encoding = pd.concat(
    [train_ml[feature_cols + [target_col]], test_ml[feature_cols + [target_col]]],
    axis=0
)

full_encoded = pd.get_dummies(
    full_for_encoding,
    columns=["gender", "subscription_type", "contract_length"],
    drop_first=True
)

train_encoded = full_encoded.iloc[:len(train_ml)].copy()
test_encoded = full_encoded.iloc[len(train_ml):].copy()

X_train = train_encoded.drop(columns=[target_col])
y_train = train_encoded[target_col]

X_test = test_encoded.drop(columns=[target_col])
y_test = test_encoded[target_col]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (440832, 12)
X_test shape: (505206, 12)


In [0]:
import pandas as pd

full_for_encoding = pd.concat(
    [train_ml[feature_cols + [target_col]], test_ml[feature_cols + [target_col]]],
    axis=0
)

full_encoded = pd.get_dummies(
    full_for_encoding,
    columns=["gender", "subscription_type", "contract_length"],
    drop_first=True
)

train_encoded = full_encoded.iloc[:len(train_ml)].copy()
test_encoded = full_encoded.iloc[len(train_ml):].copy()

X_train = train_encoded.drop(columns=[target_col])
y_train = train_encoded[target_col]

X_test = test_encoded.drop(columns=[target_col])
y_test = test_encoded[target_col]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (440832, 12)
X_test shape: (505206, 12)


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("RANDOM FOREST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, rf_preds), 4))
print("Precision:", round(precision_score(y_test, rf_preds), 4))
print("Recall:", round(recall_score(y_test, rf_preds), 4))
print("F1:", round(f1_score(y_test, rf_preds), 4))
print("ROC AUC:", round(roc_auc_score(y_test, rf_probs), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))

RANDOM FOREST RESULTS
Accuracy: 0.9285
Precision: 0.8971
Recall: 0.9841
F1: 0.9386
ROC AUC: 0.9397
Confusion Matrix:
[[193057  31657]
 [  4466 276026]]


In [0]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.head(15))

                       feature  importance
3                support_calls    0.300433
5                  total_spend    0.227847
10     contract_length_Monthly    0.141116
4                payment_delay    0.128775
0                          age    0.127480
6             last_interaction    0.035440
7                  gender_Male    0.029907
11   contract_length_Quarterly    0.005945
1                       tenure    0.001757
2              usage_frequency    0.001246
9   subscription_type_Standard    0.000035
8    subscription_type_Premium    0.000018


In [0]:
test_results = test_ml.copy()
test_results["predicted_churn"] = rf_preds
test_results["predicted_probability"] = rf_probs

print(test_results.head())

test_results = test_ml.copy()
test_results["predicted_churn"] = rf_preds
test_results["predicted_probability"] = rf_probs

print(test_results.head())

   customerid  age  gender  ...  churn  predicted_churn  predicted_probability
0           1   22  Female  ...      1                1               0.987074
1           2   41  Female  ...      0                1               0.999369
2           3   47    Male  ...      0                1               0.793986
3           4   35    Male  ...      0                1               0.996169
4           5   53  Female  ...      0                1               0.999719

[5 rows x 14 columns]
   customerid  age  gender  ...  churn  predicted_churn  predicted_probability
0           1   22  Female  ...      1                1               0.987074
1           2   41  Female  ...      0                1               0.999369
2           3   47    Male  ...      0                1               0.793986
3           4   35    Male  ...      0                1               0.996169
4           5   53  Female  ...      0                1               0.999719

[5 rows x 14 columns]


In [0]:
%sql
SELECT
  customerid,
  age,
  gender,
  tenure,
  usage_frequency,
  support_calls,
  payment_delay,
  subscription_type,
  contract_length,
  total_spend,
  last_interaction,
  churn,
  predicted_churn,
  predicted_probability
FROM workspace.default.customer_churn_predictions
ORDER BY predicted_probability DESC
LIMIT 20;

customerid,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn,predicted_churn,predicted_probability
258530,59,Female,26,21,9,29,Premium,Monthly,516,19,1,1,1.0
258496,48,Female,46,18,9,23,Standard,Annual,851,30,1,1,1.0
258476,45,Female,34,18,1,21,Standard,Annual,353,28,1,1,1.0
258488,41,Female,20,4,2,24,Basic,Monthly,964,26,1,1,1.0
258473,64,Female,23,4,2,30,Standard,Quarterly,124,5,1,1,1.0
258482,48,Male,49,28,3,28,Basic,Monthly,418,14,1,1,1.0
258477,28,Male,24,21,5,27,Premium,Monthly,596,19,1,1,1.0
258486,37,Female,44,30,0,29,Standard,Monthly,353,1,1,1,1.0
258525,37,Male,34,11,4,30,Premium,Annual,147,27,1,1,1.0
258495,56,Female,39,13,0,15,Standard,Quarterly,306,15,1,1,1.0


In [0]:
def risk_band(p):
    if p >= 0.75:
        return "High"
    elif p >= 0.40:
        return "Medium"
    else:
        return "Low"

test_results["risk_band"] = test_results["predicted_probability"].apply(risk_band)

test_results_df = spark.createDataFrame(test_results)
test_results_df.write.mode("overwrite").saveAsTable("workspace.default.customer_churn_predictions")

print(test_results[["predicted_probability", "risk_band"]].head())

   predicted_probability risk_band
0               0.987074      High
1               0.999369      High
2               0.793986      High
3               0.996169      High
4               0.999719      High


In [0]:
%sql
SELECT
  risk_band,
  COUNT(*) AS customers
FROM workspace.default.customer_churn_predictions
GROUP BY risk_band
ORDER BY customers DESC;

risk_band,customers
High,301208
Low,196290
Medium,7708


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlflow.set_experiment("/Shared/customer_churn_experiment")

with mlflow.start_run(run_name="random_forest_churn_model"):
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 8)

    acc = accuracy_score(y_test, rf_preds)
    prec = precision_score(y_test, rf_preds)
    rec = recall_score(y_test, rf_preds)
    f1 = f1_score(y_test, rf_preds)
    auc = roc_auc_score(y_test, rf_probs)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

print("MLflow run logged.")

2026/04/13 01:31:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f230962f-2b09.cloud.databricks.com/ml/experiments/3901485869553027/models/m-8a45ab96b692477aa5d576e15097033f?o=7474658483866240
2026/04/13 01:31:43 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


MLflow run logged.
